In [14]:
import pandas as pd
import numpy as np

In [15]:
match_df = pd.read_csv("raw_data/Match Dataset.csv")
ball_df = pd.read_csv("raw_data/Ball_by_Ball Dataset.csv")

print("Match Shape:", match_df.shape)
print("Ball Shape:", ball_df.shape)

Match Shape: (241, 32)
Ball Shape: (129397, 28)


In [16]:
display(match_df.head())
display(ball_df.head())

,Unnamed: 0,match_id,date,venue,city,team1,team2,toss_winner,toss_decision,winner,...,team1_player7,team2_player7,team1_player8,team2_player8,team1_player9,team2_player9,team1_player10,team2_player10,team1_player11,team2_player11
0,0,4143,2019-05-30,Kennington Oval,London,England,South Africa,1,field,0,...,MM Ali,D Pretorius,CR Woakes,AL Phehlukwayo,LE Plunkett,K Rabada,JC Archer,L Ngidi,AU Rashid,Imran Tahir
1,1,4144,2019-05-31,Trent Bridge,Nottingham,Pakistan,West Indies,1,field,1,...,Imad Wasim,JO Holder,Shadab Khan,CR Brathwaite,Hasan Ali,AR Nurse,Wahab Riaz,SS Cottrell,Mohammad Amir,O Thomas
2,2,4145,2019-06-01,Sophia Gardens,Cardiff,New Zealand,Sri Lanka,0,field,0,...,C de Grandhomme,BMAJ Mendis,MJ Santner,NLTC Perera,MJ Henry,I Udana,LH Ferguson,RAS Lakmal,TA Boult,SL Malinga
3,3,4147,2019-06-02,Kennington Oval,London,Bangladesh,South Africa,1,field,0,...,Mosaddek Hossain,AL Phehlukwayo,Mehedi Hasan Miraz,CH Morris,Mohammad Saifuddin,K Rabada,Mashrafe Mortaza,Imran Tahir,Mustafizur Rahman,L Ngidi
4,4,4148,2019-06-03,Trent Bridge,Nottingham,England,Pakistan,0,field,1,...,MM Ali,Shoaib Malik,CR Woakes,Wahab Riaz,JC Archer,Hasan Ali,AU Rashid,Shadab Khan,MA Wood,Mohammad Amir


,Unnamed: 0,match_id,inning,batting_team,bowling_team,over,ball_in_over,batter,bowler,runs_off_bat,...,balls_bowled,run_rate,target_runs,target_overs,runs_remaining,balls_remaining,required_run_rate,projected_score,pressure_index,winner
0,0,4143,1,England,South Africa,0,1,JJ Roy,Imran Tahir,1,...,1,6.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,England
1,1,4143,1,England,South Africa,0,2,JM Bairstow,Imran Tahir,0,...,2,3.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,England
2,2,4143,1,England,South Africa,0,3,JE Root,Imran Tahir,0,...,3,2.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,England
3,3,4143,1,England,South Africa,0,4,JE Root,Imran Tahir,0,...,4,1.5,NaN,NaN,NaN,NaN,NaN,0.0,NaN,England
4,4,4143,1,England,South Africa,0,5,JE Root,Imran Tahir,0,...,5,1.2,NaN,NaN,NaN,NaN,NaN,0.0,NaN,England


In [17]:
#Cleaning match_df
match_df.drop(columns=['Unnamed: 0'], inplace=True)

In [18]:
print(match_df.isnull().sum())
print()
print(match_df.dtypes)

match_id          0
date              0
venue             0
city              0
team1             0
team2             0
toss_winner       0
toss_decision     0
winner            0
team1_player1     0
team2_player1     0
team1_player2     0
team2_player2     0
team1_player3     0
team2_player3     0
team1_player4     0
team2_player4     0
team1_player5     0
team2_player5     0
team1_player6     0
team2_player6     0
team1_player7     0
team2_player7     0
team1_player8     0
team2_player8     0
team1_player9     0
team2_player9     0
team1_player10    0
team2_player10    0
team1_player11    0
team2_player11    0
dtype: int64

match_id          int64
date                str
venue               str
city                str
team1               str
team2               str
toss_winner       int64
toss_decision       str
winner            int64
team1_player1       str
team2_player1       str
team1_player2       str
team2_player2       str
team1_player3       str
team2_player3       str
team1_

In [19]:
#Data transformation
match_df['date'] = pd.to_datetime(match_df['date'])
match_df['tournament_year'] = match_df['date'].dt.year

In [20]:
match_df['toss_winner_team'] = np.where(match_df['toss_winner'] == 0,match_df['team1'],match_df['team2'])
match_df['winning_team'] = np.where(match_df['winner'] == 0,match_df['team1'],match_df['team2'])
match_df['losing_team'] = np.where(match_df['winner'] == 0,match_df['team2'],match_df['team1'])
match_df['toss_match_winner_same'] = (match_df['toss_winner_team'] == match_df['winning_team'])

In [21]:
edition_map = {
    2003: ('2003 South Africa', 'South Africa'),
    2007: ('2007 West Indies', 'West Indies'),
    2011: ('2011 India/SL/Bang', 'India'),
    2015: ('2015 Australia/NZ', 'Australia'),
    2019: ('2019 England', 'England'),
    2023: ('2023 India', 'India')
}

match_df['tournament_edition'] = match_df['tournament_year'].map(
    lambda x: edition_map.get(x, ('Unknown', 'Unknown'))[0]
)

match_df['host_nation'] = match_df['tournament_year'].map(
    lambda x: edition_map.get(x, ('Unknown', 'Unknown'))[1]
)

display(
    match_df[['tournament_year','tournament_edition','host_nation']]
    .drop_duplicates()
    .sort_values('tournament_year')
)

,tournament_year,tournament_edition,host_nation
166,2003,2003 South Africa,South Africa
73,2007,2007 West Indies,West Indies
122,2011,2011 India/SL/Bang,India
212,2015,2015 Australia/NZ,Australia
0,2019,2019 England,England
34,2023,2023 India,India


In [22]:
player_cols = [c for c in match_df.columns if 'player' in c]
match_df.drop(columns=player_cols, inplace=True)

In [23]:
#match order
match_df = match_df.sort_values(['tournament_year','date'])

match_df['match_number_in_tournament'] = (
    match_df.groupby('tournament_year').cumcount() + 1
)

In [24]:
#Ball cleaning
ball_df.drop(columns=['Unnamed: 0'], inplace=True)
ball_df['over_number'] = ball_df['over'] + 1

In [25]:
ball_df['over_phase'] = np.select(
    [
        ball_df['over_number'] <= 10,
        (ball_df['over_number'] > 10) & (ball_df['over_number'] <= 40),
        ball_df['over_number'] > 40
    ],
    ['Powerplay','Middle','Death'],
    default = 'unknown'
)

In [26]:
ball_df['is_boundary'] = (ball_df['runs_off_bat'] >= 4).astype(int)
ball_df['is_six'] = (ball_df['runs_off_bat'] == 6).astype(int)
ball_df['is_four'] = (ball_df['runs_off_bat'] == 4).astype(int)

ball_df['is_dot_ball'] = (
    (ball_df['runs_off_bat'] == 0) & (ball_df['extras'] == 0)
).astype(int)

ball_df['is_extra'] = (ball_df['extras'] > 0).astype(int)

In [27]:
#Missing Values
ball_df['wicket_type'] = ball_df['wicket_type'].fillna('not_out')
ball_df['player_out'] = ball_df['player_out'].fillna('none')
ball_df['pressure_index'] = ball_df['pressure_index'].fillna('first_innings')

In [28]:
#fill target columns
cols = ['target_runs', 'runs_remaining', 'balls_remaining']
ball_df[cols] = ball_df.groupby(['match_id', 'inning'])[cols].ffill()

In [29]:
#feature engineering
ball_df['ball_number_global'] = (ball_df['over']*6)+ball_df['ball_in_over']

ball_df['innings_label'] = np.where(
    ball_df['inning']==1,'First Innings','Second Innings'
)

In [31]:
final_df = ball_df.merge(
    match_df[
        [
            'match_id',
            'date',
            'tournament_year',
            'tournament_edition',
            'host_nation',
            'venue',
            'city',
            'team1',
            'team2',
            'toss_winner_team',
            'toss_decision',
            'winning_team',
            'losing_team',
            'toss_match_winner_same',
            'match_number_in_tournament'
        ]
    ],
    on='match_id',
    how='left'
)

print("Final Shape:", final_df.shape)
final_df.head()

Final Shape: (129397, 50)


,match_id,inning,batting_team,bowling_team,over,ball_in_over,batter,bowler,runs_off_bat,extras,...,venue,city,team1,team2,toss_winner_team,toss_decision,winning_team,losing_team,toss_match_winner_same,match_number_in_tournament
0,4143,1,England,South Africa,0,1,JJ Roy,Imran Tahir,1,0,...,Kennington Oval,London,England,South Africa,South Africa,field,England,South Africa,False,1
1,4143,1,England,South Africa,0,2,JM Bairstow,Imran Tahir,0,0,...,Kennington Oval,London,England,South Africa,South Africa,field,England,South Africa,False,1
2,4143,1,England,South Africa,0,3,JE Root,Imran Tahir,0,0,...,Kennington Oval,London,England,South Africa,South Africa,field,England,South Africa,False,1
3,4143,1,England,South Africa,0,4,JE Root,Imran Tahir,0,0,...,Kennington Oval,London,England,South Africa,South Africa,field,England,South Africa,False,1
4,4143,1,England,South Africa,0,5,JE Root,Imran Tahir,0,0,...,Kennington Oval,London,England,South Africa,South Africa,field,England,South Africa,False,1


In [32]:
final_df['match_completion_pct'] = np.where(
    final_df['inning'] == 2,
    final_df['balls_bowled'] / (final_df['target_overs'].fillna(50) * 6),
    final_df['balls_bowled'] / 300
)

final_df['match_completion_pct'] = final_df['match_completion_pct'].clip(0,1)

In [33]:
final_df.to_csv("final_worldcup_dataset.csv", index=False)
match_df.to_csv("master_matches.csv", index=False)